In [1]:
import torch
import timm

import matplotlib.pyplot as plt
import torch.nn.functional as F
import kagglehub
import os
from pathlib import Path
from PIL import Image
import numpy as np
import kagglehub
from torchvision import transforms
from torchvision import datasets
import os





In [22]:
!git clone https://github.com/nidatasneeem-prog/koa-xai-research.git

Cloning into 'koa-xai-research'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.


In [23]:
%cd koa-xai-research

/kaggle/working/koa-xai-research


In [25]:
!ls /kaggle/working

koa-xai-research


In [2]:
from pathlib import Path
import os

DATASET_DIR = Path("/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity")

print("Using dataset directory:", DATASET_DIR)

# =========================================================
# INSPECT STRUCTURE
# =========================================================
print("\nTop-level contents:\n")
for p in DATASET_DIR.iterdir():
    print(" -", p.name, "(DIR)" if p.is_dir() else "(FILE)")

# =========================================================
# FIND IMAGE FOLDERS
# =========================================================
print("\nExploring subfolders:\n")
for root, dirs, files in os.walk(DATASET_DIR):
    print(f"\n{root}")
    print(" Subfolders:", dirs)
    print(" Files:", files[:5])

Using dataset directory: /kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity

Top-level contents:

 - auto_test (DIR)
 - val (DIR)
 - test (DIR)
 - train (DIR)

Exploring subfolders:


/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity
 Subfolders: ['auto_test', 'val', 'test', 'train']
 Files: []

/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/auto_test
 Subfolders: ['2', '0', '3', '1', '4']
 Files: []

/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/auto_test/2
 Subfolders: []
 Files: ['9531901_2.png', '9768219_1.png', '9718992_2.png', '9138554_2.png', '9875303_1.png']

/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/auto_test/0
 Subfolders: []
 Files: ['9261557_2.png', '9097360_2.png', '9165552_1.png', '9337068_2.png', '9778477_1.png']

/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/auto_test/3
 Subfolders: []
 

In [4]:
from pathlib import Path
from torchvision import datasets, transforms

# define transform FIRST
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

DATASET_DIR = Path("/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity")

train_dir = DATASET_DIR / "train"
val_dir = DATASET_DIR / "val"

train_dataset = datasets.ImageFolder(
    root=str(train_dir),
    transform=transform
)



In [3]:
# =========================================================
# IMPORTS
# =========================================================
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from pathlib import Path
import cv2
import numpy as np
from PIL import Image

# =========================================================
# DEVICE
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# CLAHE TRANSFORM
# =========================================================
class CLAHETransform:
    def __call__(self, img):
        img = np.array(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(img)

        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l = clahe.apply(l)

        img = cv2.merge((l, a, b))
        img = cv2.cvtColor(img, cv2.COLOR_LAB2RGB)

        return Image.fromarray(img)

# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    CLAHETransform(),
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    CLAHETransform(),
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# =========================================================
# DATASET
# =========================================================
DATASET_DIR = Path("/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity")

train_dir = DATASET_DIR / "train"
val_dir = DATASET_DIR / "val"

train_dataset = datasets.ImageFolder(str(train_dir), transform=train_transform)
val_dataset   = datasets.ImageFolder(str(val_dir), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

# =========================================================
# CLASS WEIGHTS
# =========================================================
class_counts = {}

for class_folder in train_dir.iterdir():
    if class_folder.is_dir():
        count = len(list(class_folder.glob("*")))
        class_counts[int(class_folder.name)] = count

class_counts = dict(sorted(class_counts.items()))
counts = list(class_counts.values())

weights = 1. / torch.tensor(counts, dtype=torch.float)
class_weights = weights / weights.sum()

# =========================================================
# MODEL (DenseNet + Gradual Unfreezing)
# =========================================================
model = models.densenet121(pretrained=True)

# Freeze ALL layers first
for param in model.features.parameters():
    param.requires_grad = False

# Custom classifier
model.classifier = nn.Sequential(
    nn.Linear(model.classifier.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 5)
)

model = model.to(device)

# =========================================================
# LOSS + OPTIMIZER + SCHEDULER
# =========================================================
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.3, patience=3
)
current_lr = optimizer.param_groups[0]['lr']
print(f"Current LR: {current_lr}")
# =========================================================
# CHECKPOINT
# =========================================================
CHECKPOINT_PATH = "checkpoint.pth"
BEST_MODEL_PATH = "best_model.pth"

start_epoch = 0
best_val_acc = 0

if Path(CHECKPOINT_PATH).exists():
    print("🔁 Loading checkpoint...")

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']

    print(f"✅ Resuming from epoch {start_epoch}")

# =========================================================
# TRAIN FUNCTION
# =========================================================
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss, 100 * correct / total

# =========================================================
# VALIDATION FUNCTION
# =========================================================
def validate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total

# =========================================================
# GRADUAL UNFREEZING FUNCTION
# =========================================================
def unfreeze_layers(model, epoch):
    # Unfreeze deeper layers gradually
    if epoch == 5:
        print("🔓 Unfreezing layer block 4")
        for param in model.features.denseblock4.parameters():
            param.requires_grad = True

    if epoch == 10:
        print("🔓 Unfreezing layer block 3")
        for param in model.features.denseblock3.parameters():
            param.requires_grad = True

    if epoch == 15:
        print("🔓 Unfreezing ALL layers")
        for param in model.features.parameters():
            param.requires_grad = True

# =========================================================
# TRAIN LOOP
# =========================================================
epochs = 30

for epoch in range(start_epoch, epochs):

    # Gradual unfreezing
    unfreeze_layers(model, epoch)

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_acc = validate(model, val_loader)

    print(f"\nEpoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    # Scheduler step
    scheduler.step(val_acc)

    # Save checkpoint (resume safe)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_acc': best_val_acc
    }, CHECKPOINT_PATH)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("💾 Best model saved!")

Current LR: 0.0003

Epoch 1/30
Train Loss: 289.6089
Train Accuracy: 25.65%
Validation Accuracy: 40.92%
💾 Best model saved!

Epoch 2/30
Train Loss: 277.3700
Train Accuracy: 28.04%
Validation Accuracy: 42.86%
💾 Best model saved!

Epoch 3/30
Train Loss: 268.8530
Train Accuracy: 32.23%
Validation Accuracy: 29.54%

Epoch 4/30
Train Loss: 263.3184
Train Accuracy: 31.93%
Validation Accuracy: 36.44%

Epoch 5/30
Train Loss: 257.4744
Train Accuracy: 32.68%
Validation Accuracy: 41.28%
🔓 Unfreezing layer block 4

Epoch 6/30
Train Loss: 238.4805
Train Accuracy: 37.04%
Validation Accuracy: 46.61%
💾 Best model saved!

Epoch 7/30
Train Loss: 209.1167
Train Accuracy: 45.28%
Validation Accuracy: 44.19%

Epoch 8/30
Train Loss: 200.0267
Train Accuracy: 46.18%
Validation Accuracy: 51.33%
💾 Best model saved!

Epoch 9/30
Train Loss: 195.5273
Train Accuracy: 47.72%
Validation Accuracy: 51.21%

Epoch 10/30
Train Loss: 192.5356
Train Accuracy: 49.00%
Validation Accuracy: 52.91%
💾 Best model saved!
🔓 Unfreezing 

In [4]:
# =========================================================
# LOAD CHECKPOINT
# =========================================================
checkpoint = torch.load("checkpoint.pth", map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch'] + 1
best_val_acc = checkpoint['best_val_acc']

print(f"✅ Resuming from epoch {start_epoch}")

# =========================================================
# CONTINUE TRAINING UNTIL 70
# =========================================================
total_epochs = 70   # 👈 FINAL TARGET

for epoch in range(start_epoch, total_epochs):

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_acc = validate(model, val_loader)

    print(f"\nEpoch {epoch+1}/{total_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    scheduler.step(val_acc)

    # Save checkpoint (for crash/resume)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_acc': best_val_acc
    }, "checkpoint.pth")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("💾 Best model saved!")

✅ Resuming from epoch 30

Epoch 31/70
Train Loss: 104.0655
Train Accuracy: 69.59%
Validation Accuracy: 64.41%

Epoch 32/70
Train Loss: 100.9025
Train Accuracy: 70.99%
Validation Accuracy: 65.50%

Epoch 33/70
Train Loss: 98.4352
Train Accuracy: 71.95%
Validation Accuracy: 66.71%

Epoch 34/70
Train Loss: 98.4573
Train Accuracy: 71.91%
Validation Accuracy: 64.65%

Epoch 35/70
Train Loss: 98.6716
Train Accuracy: 71.98%
Validation Accuracy: 66.10%

Epoch 36/70
Train Loss: 98.4463
Train Accuracy: 71.55%
Validation Accuracy: 65.98%

Epoch 37/70
Train Loss: 98.9294
Train Accuracy: 71.65%
Validation Accuracy: 66.46%

Epoch 38/70
Train Loss: 96.3870
Train Accuracy: 72.14%
Validation Accuracy: 65.98%

Epoch 39/70
Train Loss: 98.5503
Train Accuracy: 72.08%
Validation Accuracy: 66.34%

Epoch 40/70
Train Loss: 95.9515
Train Accuracy: 72.88%
Validation Accuracy: 66.95%
💾 Best model saved!

Epoch 41/70
Train Loss: 96.5635
Train Accuracy: 71.96%
Validation Accuracy: 65.50%

Epoch 42/70
Train Loss: 95.3

In [5]:
# =========================================================
# LOAD BEST MODEL (IMPORTANT)
# =========================================================
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model = model.to(device)

print("✅ Loaded best model (epoch ~57)")

# =========================================================
# UNFREEZE ALL LAYERS
# =========================================================
for param in model.parameters():
    param.requires_grad = True

print("🔓 All layers unfrozen")

# =========================================================
# NEW OPTIMIZER (LOW LR)
# =========================================================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

# Optional: scheduler (still useful)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.3, patience=2
)

# =========================================================
# CONTINUE TRAINING (FINE-TUNING)
# =========================================================
start_epoch = 70          # where you left off
total_epochs = 80         # train 10 more epochs

best_val_acc = 0          # will reload from checkpoint if exists

# Try loading checkpoint for continuity
if Path("checkpoint.pth").exists():
    checkpoint = torch.load("checkpoint.pth", map_location=device)
    best_val_acc = checkpoint['best_val_acc']

    print(f"📊 Previous best validation accuracy: {best_val_acc:.2f}%")

# =========================================================
# TRAIN LOOP (REFINEMENT)
# =========================================================
for epoch in range(start_epoch, total_epochs):

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_acc = validate(model, val_loader)

    print(f"\nEpoch {epoch+1}/{total_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    scheduler.step(val_acc)

    # Save checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_acc': best_val_acc
    }, "checkpoint.pth")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("💾 New BEST model saved!")

✅ Loaded best model (epoch ~57)
🔓 All layers unfrozen
📊 Previous best validation accuracy: 67.31%

Epoch 71/80
Train Loss: 97.9272
Train Accuracy: 71.81%
Validation Accuracy: 66.46%

Epoch 72/80
Train Loss: 94.6908
Train Accuracy: 71.20%
Validation Accuracy: 66.10%

Epoch 73/80
Train Loss: 94.0326
Train Accuracy: 73.16%
Validation Accuracy: 66.34%

Epoch 74/80
Train Loss: 95.0952
Train Accuracy: 72.60%
Validation Accuracy: 66.10%

Epoch 75/80
Train Loss: 93.4054
Train Accuracy: 72.40%
Validation Accuracy: 66.59%

Epoch 76/80
Train Loss: 91.9794
Train Accuracy: 73.94%
Validation Accuracy: 66.46%

Epoch 77/80
Train Loss: 92.6322
Train Accuracy: 73.75%
Validation Accuracy: 66.46%

Epoch 78/80
Train Loss: 93.6591
Train Accuracy: 72.88%
Validation Accuracy: 66.71%

Epoch 79/80
Train Loss: 91.7366
Train Accuracy: 73.28%
Validation Accuracy: 66.59%

Epoch 80/80
Train Loss: 92.8701
Train Accuracy: 72.79%
Validation Accuracy: 64.89%
